# Imports

In [1]:
# Import necessary libraries for Google Colab
import os
from google.colab import drive

# Mount Google Drive (if your files are on Google Drive)
drive.mount('/content/drive')

# Change directory to your project folder
os.chdir('/content/drive/MyDrive/Colab Notebooks/multitask_package/multitask_basic/')

Mounted at /content/drive


In [2]:
# Install dependencies
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 15.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 72.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.3/106.3 kB 13.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.2/241.2 kB 25.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 51.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 39.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 74.8 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=5692a141645f48d726d89061669d65eee405833984c0fb651c1ae8932238739a
 

## Imports

In [3]:
"""# Dependencies """

import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
import json
import tensorflow_text as text

from utils.data_utils import prepare_data
from train_test.train_test import train
from train_test.train_test import test
from utils.encoder_utils import get_preprocessor
from models.model import BertClassifier

import argparse
import yaml

# Setup

## GPU setup

In [4]:
GPU = tf.config.list_physical_devices("GPU")
for gpu in GPU:
    tf.config.experimental.set_memory_growth(gpu, True)

strategy = tf.distribute.MirroredStrategy()

## Arguments, data, and Target tasks

In [5]:
config_file = './configs/example_cmp.yaml'
with open(config_file, 'r') as file:
    config = yaml.safe_load(file)

In [6]:
main_task_names = config['main_task_names']
task_sizes = config['task_sizes']
pooled_task_name = config['pooled_task_name']
"""
targets should be a dictionary that is read from a json
targets = {'202.1':'202.1', '202.3':'202.3', '202.4':'202.4',
        '305.1':'305.1', '305.2':'305.2', '305.3':'305.3',
        '607.1':'607.1', '607.2':'607.2', '607.3':'607.3',
        '608.2':'608.2', '608.3':'608.3'}
"""
with open(config['targets_file'], 'r') as fin:
    targets = json.load(fin)
"""# Loading data"""
#read in training and target data and remove leading column
"""
select_columns = ['text',
                '202.1', '202.3', '202.4',
                '305.1', '305.2', '305.3',
                '607.1', '607.2', '607.3',
                '608.2', '608.3']
"""
select_columns = config['select_columns']
num_tasks = len(select_columns) - 1
data_path = config['data_path']


ff_train, ff_val, ff_test = prepare_data(data_path, select_columns, task_sizes, main_task_names,
                                        sample_size_ratio=config['sample_size_ratio'],
                                        data_type="int16")

task_list=[target_name.replace("_bi", "" ) for target_name in list(ff_train.columns)[1:]]

#Training

## Training parameters

In [7]:
print("Specifying model settings and instantiating preprocessor...")
preprocessor, encoder = get_preprocessor(encoder_type=config['encoder']['encoder_type'],
                                        model_name=config['encoder']['encoder_model_name'])


#specify training settings

seq_length = tf.constant(512) #cannot be greater than 512
workers = 1 #greater than 1 if using MultiWorkerMirroredStrategy

#define list of parameters for grid search

# --- grid search params
# pars_grid = [(init_lr, wt_decay_rate, warmup_pct, dropout, batch_size)
#              for init_lr in [15e-5, 3e-4]
#              for wt_decay_rate in [0.01, 0.05, 0.1]
#              for warmup_pct in [5, 20]
#              for dropout in [0.1, 0.25, 0.4]
#              for batch_size in [32, 64]]

# best parameters found from the above grid search
params_dict = config['training_params']
epochs, init_lr = params_dict['epochs'], params_dict['init_lr']
wt_decay_rate, warmup_pct = params_dict['wt_decay_rate'], params_dict['warmup_pct']
dropout, batch_size = params_dict['dropout'], params_dict['batch_size']
workers, seq_length = params_dict['workers'], params_dict['seq_length']
#params = [init_lr, wt_decay_rate, warmup_pct, dropout, batch_size, epochs]


# Create an instance of the model
with strategy.scope():
    bert_classifier = BertClassifier(num_classes=2, dropout=dropout,
                                    task_target_sizes=task_sizes,
                                    main_task_names=main_task_names,
                                    encoder=encoder)

Specifying model settings and instantiating preprocessor...


## Train

In [8]:
bert_classifier, pars_best = train(strategy, bert_classifier,
                                  ff_train, ff_val,
                                  task_sizes, main_task_names, num_tasks,
                                  preprocessor,
                                  epochs=epochs, init_lr=init_lr, wt_decay_rate=wt_decay_rate,
                                  warmup_pct=warmup_pct, dropout=dropout, batch_size=batch_size,
                                  workers=workers, seq_length=seq_length)


"""### Saving parameters and thresholds"""
import pickle
import datetime
fname_time = '{date:%Y_%m_%d_%H_%M_%S}'.format(date=datetime.datetime.now())

fname_prefix = './'

with open(f'{fname_prefix}{config["dataset_name"]}_pars_best_{fname_time}.pkl', 'wb') as fout:
    pickle.dump(pars_best, fout, pickle.HIGHEST_PROTOCOL)

Training init_lr = 0.0003, wt_decay_rate = 0.01, warmup_pct = 5, dropout = 0.4, batch_size = 256...
Epoch 1/3
4/4 [==============================] - 188s 45s/step - loss: 0.0090 - accuracy: 0.6299 - precision: 0.0051 - recall: 0.2800 - val_loss: 0.0089 - val_accuracy: 0.6307 - val_precision: 0.0058 - val_recall: 0.3333
Epoch 2/3
2/2 [==============================] - 24s 11s/step
Val weighted f1: 0.8066594282352635


# Testing

In [9]:
metrics_dict, metrics_str = test(strategy, bert_classifier,
                                ff_test, pars_best,
                                task_list, task_sizes, main_task_names, num_tasks,
                                preprocessor,
                                pooled_task_name=pooled_task_name,
                                batch_size=batch_size,
                                workers=workers, seq_length=seq_length)


"""### Saving performance metrics"""

fname_metrics = f"./results/metrics_{config['dataset_name']}_{fname_time}.json"
with open(fname_metrics, "w") as fout:
    json.dump(metrics_dict, fout)
print(f"\n ====== Testing metrics saved in {fname_metrics} ====== \n")

fname_report = f"./results/output_{config['dataset_name']}_{fname_time}.txt"
with open(fname_report, 'w') as fout:
    fout.write(metrics_str)
print(f"\n ====== Testing report saved in {fname_report} ====== \n")

2/2 [==============================] - 21s 9s/step
202.1
AUC: 0.5204918032786885
              precision    recall  f1-score   support

           0      0.978     0.541     0.697       488
           1      0.026     0.500     0.050        12

    accuracy                          0.540       500
   macro avg      0.502     0.520     0.373       500
weighted avg      0.955     0.540     0.681       500

202.3
AUC: undefined
              precision    recall  f1-score   support

           0      1.000     0.906     0.951       500
           1      0.000     0.000     0.000         0

    accuracy                          0.906       500
   macro avg      0.500     0.453     0.475       500
weighted avg      1.000     0.906     0.951       500

202.4
AUC: 0.6563126252505009
              precision    recall  f1-score   support

           0      1.000     0.313     0.476       499
           1      0.003     1.000     0.006         1

    accuracy                          0.314       